In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
len(documents)

72

In [3]:
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"

!wget {PREFIX}/01-agentic-rag/code/rag_helper.py
!wget {PREFIX}/04-evaluation/code/evaluation_utils.py

--2026-07-09 19:21:33--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-07-09 19:21:33 (15.2 MB/s) - ‘rag_helper.py’ saved [2134/2134]

--2026-07-09 19:21:33--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response.

In [7]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [13]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [14]:
import json
user_prompt = json.dumps(doc)

In [11]:
from pydantic import BaseModel
from evaluation_utils import llm_structured, llm_structured_retry

class Questions(BaseModel):
    questions: list[str]

def generate_ground_truth(doc):
    user_prompt = json.dumps({
        "filename": doc["filename"],
        "content": doc["content"],
    })

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
    )
    records = []
    for q in out.questions:
        records.append({
            "question": q,
            "filename": doc["filename"],
        })

    return records, usage

In [15]:
first_three = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]

docs_by_filename = {doc["filename"]: doc for doc in documents}

usages = []
for filename in first_three:
    doc = docs_by_filename[filename]
    user_prompt = json.dumps({"filename": doc["filename"], "content": doc["content"]})
    _, usage = llm_structured(openai_client, data_gen_instructions, user_prompt, Questions)
    usages.append(usage)
    print(filename, usage.input_tokens)

avg_input_tokens = sum(u.input_tokens for u in usages) / len(usages)
print("Q1 average input tokens:", avg_input_tokens)

01-agentic-rag/lessons/01-intro.md 1021
01-agentic-rag/lessons/02-environment.md 1287
01-agentic-rag/lessons/03-rag.md 1754
Q1 average input tokens: 1354.0


In [17]:
import pandas as pd

In [18]:
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"

!wget -O ground-truth.csv {PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")
len(ground_truth)  # 360

--2026-07-09 19:40:58--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 48627 (47K) [text/plain]
Saving to: ‘ground-truth.csv’

ground-truth.csv    100%[===================>]  47.49K  --.-KB/s    in 0.07s   

2026-07-09 19:41:01 (659 KB/s) - ‘ground-truth.csv’ saved [48627/48627]



360